# Baltic SST — Yearly Comparison

This script reads Baltic SST from Copernicus Marine, computes monthly means per year,
and produces two interactive plots:
1. **Spatial map** — monthly mean SST with a time slider, per year
2. **Seasonal cycle** — area-mean SST lines overlaid for each year

`version 1.0.2` `23.04.2026`

In [1]:
import copernicusmarine
import hvplot.xarray  # noqa
import warnings

warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

## Open dataset (lazy)

In [2]:
dataset_id = 'DMI-BALTIC-SST-L4-NRT-OBS_FULL_TIME_SERIE'
ds = copernicusmarine.open_dataset(dataset_id)
ds

INFO - 2026-04-23T10:02:55Z - Selected dataset version: "202511"
INFO - 2026-04-23T10:02:55Z - Selected dataset part: "default"


<xarray.Dataset> Size: 60GB
Dimensions:           (time: 1180, latitude: 901, longitude: 2001)
Coordinates:
  * latitude          (latitude) float32 4kB 48.0 48.02 48.04 ... 65.98 66.0
  * longitude         (longitude) float32 8kB -10.0 -9.98 -9.96 ... 29.98 30.0
  * time              (time) datetime64[ns] 9kB 2023-01-30 ... 2026-04-23
Data variables:
    analysed_sst      (time, latitude, longitude) float64 17GB dask.array<chunksize=(506, 128, 2001), meta=np.ndarray>
    analysis_error    (time, latitude, longitude) float64 17GB dask.array<chunksize=(506, 128, 2001), meta=np.ndarray>
    mask              (time, latitude, longitude) float32 9GB dask.array<chunksize=(506, 128, 2001), meta=np.ndarray>
    sea_ice_fraction  (time, latitude, longitude) float64 17GB dask.array<chunksize=(506, 128, 2001), meta=np.ndarray>
Attributes: (12/48)
    Conventions:                CF-1.4, Unidata Observation Dataset v1.0
    Metadata_Conventions:       Unidata Dataset Discovery v1.0
    acknowledgment:             Please acknowledge the use of these data with...
    cdm_data_type:              grid
    comment:                    IN NO EVENT SHALL DMI OR ITS REPRESENTATIVES ...
    creator_email:              jlh@dmi.dk
    ...                         ...
    time_coverage_end:          20251111T000000Z
    time_coverage_start:        20251110T000000Z
    title:                      DMI Sea Surface Temperature analysis
    uuid:                       a1efd321-d8bd-4e12-8c6d-fae3f51adcfb
    westernmost_longitude:      -10.0
    copernicusmarine_version:   2.3.0

## Spatial subset to the Baltic Sea

The Baltic proper sits roughly `lat 54–66 °N, lon 10–30 °E`.
Using `time=-1` in the chunk keeps all timesteps in a single chunk along
the time axis so that `resample().mean()` can scan each spatial tile in
one pass instead of assembling thousands of tiny cross-chunk tasks.

In [3]:
ds

<xarray.Dataset> Size: 60GB
Dimensions:           (time: 1180, latitude: 901, longitude: 2001)
Coordinates:
  * latitude          (latitude) float32 4kB 48.0 48.02 48.04 ... 65.98 66.0
  * longitude         (longitude) float32 8kB -10.0 -9.98 -9.96 ... 29.98 30.0
  * time              (time) datetime64[ns] 9kB 2023-01-30 ... 2026-04-23
Data variables:
    analysed_sst      (time, latitude, longitude) float64 17GB dask.array<chunksize=(506, 128, 2001), meta=np.ndarray>
    analysis_error    (time, latitude, longitude) float64 17GB dask.array<chunksize=(506, 128, 2001), meta=np.ndarray>
    mask              (time, latitude, longitude) float32 9GB dask.array<chunksize=(506, 128, 2001), meta=np.ndarray>
    sea_ice_fraction  (time, latitude, longitude) float64 17GB dask.array<chunksize=(506, 128, 2001), meta=np.ndarray>
Attributes: (12/48)
    Conventions:                CF-1.4, Unidata Observation Dataset v1.0
    Metadata_Conventions:       Unidata Dataset Discovery v1.0
    acknowledgment:             Please acknowledge the use of these data with...
    cdm_data_type:              grid
    comment:                    IN NO EVENT SHALL DMI OR ITS REPRESENTATIVES ...
    creator_email:              jlh@dmi.dk
    ...                         ...
    time_coverage_end:          20251111T000000Z
    time_coverage_start:        20251110T000000Z
    title:                      DMI Sea Surface Temperature analysis
    uuid:                       a1efd321-d8bd-4e12-8c6d-fae3f51adcfb
    westernmost_longitude:      -10.0
    copernicusmarine_version:   2.3.0

In [4]:
# ds_baltic = (ds.sel(latitude=slice(54, 66), longitude=slice(8, 30)).chunk({'time': 100, 'latitude': 100, 'longitude': 100}))
ds_baltic = (ds.sel(latitude=slice(54, 66), longitude=slice(8, 30)))
ds_baltic

<xarray.Dataset> Size: 22GB
Dimensions:           (time: 1180, latitude: 601, longitude: 1100)
Coordinates:
  * latitude          (latitude) float32 2kB 54.0 54.02 54.04 ... 65.98 66.0
  * longitude         (longitude) float32 4kB 8.02 8.04 8.06 ... 29.98 30.0
  * time              (time) datetime64[ns] 9kB 2023-01-30 ... 2026-04-23
Data variables:
    analysed_sst      (time, latitude, longitude) float64 6GB dask.array<chunksize=(506, 84, 1100), meta=np.ndarray>
    analysis_error    (time, latitude, longitude) float64 6GB dask.array<chunksize=(506, 84, 1100), meta=np.ndarray>
    mask              (time, latitude, longitude) float32 3GB dask.array<chunksize=(506, 84, 1100), meta=np.ndarray>
    sea_ice_fraction  (time, latitude, longitude) float64 6GB dask.array<chunksize=(506, 84, 1100), meta=np.ndarray>
Attributes: (12/48)
    Conventions:                CF-1.4, Unidata Observation Dataset v1.0
    Metadata_Conventions:       Unidata Dataset Discovery v1.0
    acknowledgment:             Please acknowledge the use of these data with...
    cdm_data_type:              grid
    comment:                    IN NO EVENT SHALL DMI OR ITS REPRESENTATIVES ...
    creator_email:              jlh@dmi.dk
    ...                         ...
    time_coverage_end:          20251111T000000Z
    time_coverage_start:        20251110T000000Z
    title:                      DMI Sea Surface Temperature analysis
    uuid:                       a1efd321-d8bd-4e12-8c6d-fae3f51adcfb
    westernmost_longitude:      -10.0
    copernicusmarine_version:   2.3.0

In [5]:
#cluster_type = 'Local'    
cluster_type = 'Coiled'
# cluster_type = 'Gateway'
#cluster_type = 'Coiled'

In [6]:
if cluster_type == 'Coiled':
    import coiled
    region="eu-central-1"
    cluster = coiled.Cluster(
        region=region,
        arm=True,   # run on ARM to save energy & cost
        worker_vm_types=["t4g.large"],  # cheap, small ARM instances, 2cpus, 2GB RAM 
        # worker_options={'nthreads':2}, # Amazon does not charge for this
        n_workers=30,
        name=f'moeindst_{region}',
        wait_for_workers=False,
        compute_purchase_option="spot_with_fallback",
        software='protocoast-notebook-arm',  # Conda environment name
        workspace='esip-lab',
        timeout=180   # leave cluster running for 3 min in case we want to use it again
    )

    client = cluster.get_client()

Output()

2026-04-23 10:25:10,585 - distributed.client - ERROR - Failed to reconnect to scheduler after 30.00 seconds, closing client


In [6]:
client

<Client: 'tls://10.0.25.127:8786' processes=29 threads=58, memory=207.69 GiB>

## Compute monthly means and convert to °C

Resample once on the full time axis, load the small result (~200 MB total),
then split by year — one `.load()` call for all years combined.

In [7]:
%%time
da_monthly = (
    ds_baltic['analysed_sst']
    .resample(time='ME')   # 'ME' = month-end
    .mean()
    .load()
) - 273.15                 # kelvin to celcius

da_monthly.attrs['units'] = '°C'
# da_monthly

CPU times: user 1.15 s, sys: 779 ms, total: 1.93 s
Wall time: 17.7 s


## Split by year

In [8]:
years = sorted({int(t) for t in da_monthly.time.dt.year.values})
da_by_year = {yr: da_monthly.sel(time=str(yr)) for yr in years}
print({yr: da.sizes for yr, da in da_by_year.items()})

{2023: Frozen({'time': 12, 'latitude': 601, 'longitude': 1100}), 2024: Frozen({'time': 12, 'latitude': 601, 'longitude': 1100}), 2025: Frozen({'time': 12, 'latitude': 601, 'longitude': 1100}), 2026: Frozen({'time': 4, 'latitude': 601, 'longitude': 1100})}


## Plot 1 — Spatial monthly SST maps with time slider

One panel per year, each with a month slider via `groupby='time'`.
Plots are laid out in two columns using `+` and `.cols(2)`.

In [11]:
map_opts = dict(                                                                                         
      x='longitude', y='latitude',                                                                         
      rasterize=True, geo=True,                                                                            
      cmap='turbo', clim=(-2, 25),                                                                         
      tiles='OSM',                                                                                         
      width=700, height=500,                                                                               
  )                                                                                                        

plot1 = da_monthly.hvplot(title='Baltic SST Monthly Mean (°C)', groupby='time', **map_opts)
# plot1 = da_monthly.hvplot(title='Baltic SST Monthly Mean (°C)', **map_opts)                             
plot1

:DynamicMap   [time]
   :Overlay
      .WMTS.I  :WMTS   [Longitude,Latitude]
      .Image.I :Image   [longitude,latitude]   (analysed_sst)

## Plot 2 — Seasonal cycle: area-mean SST per year, all overlaid

Compute the Baltic-wide spatial mean for each monthly snapshot,
then plot all years as curves on the same axes using `*` to overlay.

In [13]:
import numpy as np
import pandas as pd

# Build a tidy DataFrame: month (1-12) vs year columns
records = {}
for yr, da in da_by_year.items():
    area_mean = da.mean(dim=['latitude', 'longitude']).values
    months = da.time.dt.month.values
    records[str(yr)] = pd.Series(area_mean, index=months)

df_seasonal = pd.DataFrame(records)
df_seasonal.index.name = 'month'
df_seasonal

,2023,2024,2025,2026
month,,,,
1,3.019102,2.209291,3.560737,3.648840
2,2.572890,1.658099,2.405146,1.153111
3,2.124799,1.732728,2.699695,1.897923
4,3.353136,2.897240,4.457954,3.467697
5,7.321849,8.397325,7.287622,NaN
6,13.707028,14.401626,11.904762,NaN
7,16.733578,17.168677,16.994919,NaN
8,17.183603,18.322450,18.357154,NaN
9,16.410846,16.540729,16.058658,NaN


In [14]:
import hvplot.pandas  # noqa
import operator
from functools import reduce

COLORS = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']

curves = [
    df_seasonal[col].hvplot.line(
        label=col,
        color=COLORS[i % len(COLORS)],
        line_width=2,
    )
    for i, col in enumerate(df_seasonal.columns)
]

seasonal_plot = reduce(operator.mul, curves).opts(
    title='Baltic SST — Monthly Area-Mean by Year',
    xlabel='Month', ylabel='SST (°C)',
    legend_position='top_left',
    width=700, height=400,
    show_grid=True,
)
seasonal_plot

:Overlay
   .Curve.A_2023 :Curve   [month]   (2023)
   .Curve.A_2024 :Curve   [month]   (2024)
   .Curve.A_2025 :Curve   [month]   (2025)
   .Curve.A_2026 :Curve   [month]   (2026)

                                        ----------- End of Document -----------